# Regime Examples Figure — rebuilt with corrected 2024 data

Rebuilds `fig:different_types_trajectories` using the SAME four chosen sequences
(one per activity/dynamism regime) as `paper_regime_examples_figure.ipynb`, but reading
frames from the newly-downloaded `pickled_maps_new.zip` (2024) instead of the local
`pickled_maps` directory.

**Why:** the local `pickled_maps` (2024) copy was found to differ from every other data
source (`all_maps` on scratch, and all `pickled_maps_new_*.zip` archives for 2020-2025) by
a rotation along the longitude axis that varies linearly with UT (~15 deg/hour = Earth's
rotation rate) — i.e. it is very likely mis-rotated. `pickled_maps_new.zip` agrees byte-for-byte
with `all_maps` for both 2020 and 2025 spot checks, so it is treated here as the correct source.

The four start times below are reused as-is from the original notebook's own candidate search
(saved cell outputs), so only the data source changes, not the example selection.

In [1]:
import sys, os
sys.path.insert(0, '/users/framunno/projects/ionosphere_diffusion')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import map_coordinates
from src.data.dataset import latlon_to_cartesian_grid

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'STIXGeneral'

PAPER_FIGURES_DIR = '/users/framunno/projects/paper_writing/SuperDARN_deep_Learning_Francesco/figures'
NEW_ZIP_2024 = '/capstor/scratch/cscs/framunno/ionosphere_data/pickled_maps_new.zip'
CACHE_DIR = '/users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024'

REGIMES = ['dynhigh_acthigh', 'dynhigh_actlow', 'dynlow_acthigh', 'dynlow_actlow']
REGIME_LABELS = {
    'dynhigh_acthigh': 'high activity / high dynamism',
    'dynhigh_actlow':  'low activity / high dynamism',
    'dynlow_acthigh':  'high activity / low dynamism',
    'dynlow_actlow':   'low activity / low dynamism',
}
# Same start times chosen by the original notebook's candidate search (see markdown above)
STARTS = {
    'dynhigh_acthigh': '2024-10-10 19:00:00',
    'dynhigh_actlow':  '2024-10-28 01:42:00',
    'dynlow_acthigh':  '2024-10-11 09:24:00',
    'dynlow_actlow':   '2024-10-14 00:14:00',
}

In [2]:
import subprocess, pathlib

os.makedirs(CACHE_DIR, exist_ok=True)

def needed_filenames(start_str, n_frames=22, step_min=2):
    start = pd.Timestamp(start_str)
    times = [start + pd.Timedelta(minutes=step_min*i) for i in range(n_frames)]
    return times, [f"map_{t.year}_{t.month}_{t.day}_{t.hour}_{t.minute}_0.npy" for t in times]

all_names = set()
per_regime_times = {}
for r in REGIMES:
    times, names = needed_filenames(STARTS[r])
    per_regime_times[r] = times
    all_names.update(names)

missing = [n for n in all_names if not (pathlib.Path(CACHE_DIR) / n).exists()]
print(f'{len(all_names)} unique frames needed, {len(missing)} not yet cached locally')

if missing:
    patterns = [f'*{n}' for n in missing]
    subprocess.run(['unzip', '-j', '-o', NEW_ZIP_2024, *patterns, '-d', CACHE_DIR], check=True)
print('cache dir now has', len(list(pathlib.Path(CACHE_DIR).glob('*.npy'))), 'files')

88 unique frames needed, 88 not yet cached locally
Archive:  /capstor/scratch/cscs/framunno/ionosphere_data/pickled_maps_new.zip


  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_0_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_10_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_12_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_14_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_16_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_18_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_20_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_202

  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_30_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_32_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_34_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_36_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_38_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_40_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_10_19_42_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_20

  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_28_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_30_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_32_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_34_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_36_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_38_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_11_9_40_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_


  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_16_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_18_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_20_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_22_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_24_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_26_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_28_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map

  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_40_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_42_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_44_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_46_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_48_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_50_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_14_0_52_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_


  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_1_58_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_2_0_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_2_10_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_2_12_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_2_14_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_2_16_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_2024_10_28_2_18_0.npy  
  inflating: /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/regime_cache_2024/map_

In [3]:
chosen = {}
for r in REGIMES:
    frames = []
    for t in per_regime_times[r]:
        fname = f"map_{t.year}_{t.month}_{t.day}_{t.hour}_{t.minute}_0.npy"
        raw = np.load(os.path.join(CACHE_DIR, fname), allow_pickle=True)[0].astype(np.float32)
        frames.append(latlon_to_cartesian_grid(raw, output_size=128))
    chosen[r] = {'frames': np.stack(frames), 'times': per_regime_times[r]}
    print(r, '-> start', per_regime_times[r][0], ', loaded', len(frames), 'frames')

dynhigh_acthigh -> start 2024-10-10 19:00:00 , loaded 22 frames


dynhigh_actlow -> start 2024-10-28 01:42:00 , loaded 22 frames


dynlow_acthigh -> start 2024-10-11 09:24:00 , loaded 22 frames


dynlow_actlow -> start 2024-10-14 00:14:00 , loaded 22 frames


In [4]:
H_IMG, MAX_R = 128, 24
r_i, th_i = np.linspace(0, MAX_R, 200), np.linspace(0, 2*np.pi, 360)
r_grid, theta_grid = np.meshgrid(r_i, th_i)
polar_x, polar_y = r_grid*np.cos(theta_grid), r_grid*np.sin(theta_grid)
col_coords = (polar_x + MAX_R) / (2*MAX_R) * (H_IMG - 1)
row_coords = (polar_y + MAX_R) / (2*MAX_R) * (H_IMG - 1)

def to_polar(frame2d):
    return map_coordinates(frame2d, [row_coords, col_coords], order=1, mode='constant', cval=0)

VMAX = max(np.abs(chosen[r]['frames']).max() for r in REGIMES)
print('shared VMAX:', VMAX)

shared VMAX: 41967.72657842311


In [5]:
N_SNAPSHOTS = 7
snapshot_idx = np.linspace(0, 21, N_SNAPSHOTS).round().astype(int)

fig = plt.figure(figsize=(14.5, 11))
gs_outer = gridspec.GridSpec(4, 1, hspace=0.65, left=0.06, right=0.90, top=0.92, bottom=0.05, figure=fig)

last_mesh = None
for ri, r in enumerate(REGIMES):
    data = chosen[r]
    gs_row = gridspec.GridSpecFromSubplotSpec(1, N_SNAPSHOTS, subplot_spec=gs_outer[ri], wspace=0.12)
    for ci, fi in enumerate(snapshot_idx):
        ax = fig.add_subplot(gs_row[ci], projection='polar')
        last_mesh = ax.pcolormesh(theta_grid, r_grid, to_polar(data['frames'][fi]), shading='auto',
                                   cmap='coolwarm', vmin=-VMAX, vmax=VMAX)
        ax.set_theta_zero_location('S')
        ax.set_theta_direction(1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.spines['polar'].set_linewidth(0.8)
        ax.spines['polar'].set_color('0.55')

    fig.canvas.draw()
    row_axes = fig.axes[-N_SNAPSHOTS:]
    x_mid = (row_axes[0].get_position().x0 + row_axes[-1].get_position().x1) / 2
    y_top = row_axes[0].get_position().y1 + 0.02
    fig.text(x_mid, y_top, REGIME_LABELS[r], fontsize=16, fontweight='bold', ha='center', va='bottom',
              fontfamily='STIXGeneral')

cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(last_mesh, cax=cbar_ax)
cbar.set_label('Electric potential [V]', fontsize=18, fontfamily='STIXGeneral')
cbar.ax.tick_params(labelsize=15)

from matplotlib.patches import FancyArrowPatch
bottom_row_axes = fig.axes[-N_SNAPSHOTS-1:-1]
x0 = bottom_row_axes[0].get_position().x0
x1 = bottom_row_axes[-1].get_position().x1
y_arrow = bottom_row_axes[0].get_position().y0 - 0.045
arrow = FancyArrowPatch((x0, y_arrow), (x1, y_arrow), transform=fig.transFigure,
                         arrowstyle='-|>', mutation_scale=16, color='0.25', lw=1.4)
fig.add_artist(arrow)
fig.text((x0 + x1) / 2, y_arrow - 0.025, 'Time', fontsize=14, ha='center', va='top', fontfamily='STIXGeneral')

OUT = '/users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/methodology_regime_examples_REDONE_new_data.png'
fig.savefig(OUT, dpi=200, bbox_inches='tight')
print('saved', OUT)
plt.show()

/tmp/ipykernel_211631/2744443894.py:13: MatplotlibDeprecationWarning: Auto-removal of grids by pcolor() and pcolormesh() is deprecated since 3.5 and will be removed two minor releases later; please call grid(False) first.
  last_mesh = ax.pcolormesh(theta_grid, r_grid, to_polar(data['frames'][fi]), shading='auto',


saved /users/framunno/projects/ionosphere_diffusion/notebooks/figures_diag/methodology_regime_examples_REDONE_new_data.png
